In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import os
from SRNN import model_srnn
from SRNN import inference_network
from SRNN import initialization
from SRNN import train
from SRNN import generative_check
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(131)
torch.manual_seed(131)

"""### Step 2: Data load
In this step, we load and visualize the data of lorenz attractor.
"""
jobid= 5

sim_out=np.load('./data/act_save_single_200.npy',allow_pickle=True)[int(jobid)]

y_c=sim_out/sim_out.max()

"""We then split the data into training and testing, i.e., 17 trials in training and 1 trials in testing. 'jobid' is to set which trial in testing."""


train_data, test_data = train_test_split(y_c, test_size=0.2, random_state=42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32
X_train=torch.tensor(0*train_data,dtype=dtype,device=device)
y_train=torch.tensor(train_data,dtype=dtype,device=device)
y_test=torch.tensor(test_data,dtype=dtype,device=device)
X_test=torch.tensor(0*test_data,dtype=dtype,device=device)
### beh_all_train and beh_all_test are behavioral labels, we don't have to use them in lorenz attractor.
beh_all_train=None
beh_all_test=None

"""### Step 3: Hyperparameters"""

input_shape=X_train.shape[2] # Input shape of SRNNs, but the models are input free.
num_tv=2 # Number of RNNs in SRNNs.
hidden_shape=8 # Number of hidden states of SRNNs.
ini_epochs=3000 # Epochs in initialization stage, can be longer than training stage.
coef_cross=50e-1 # Coefficient of initialization, larger coef_cross means larger constraint on posterior states in initialization.
epochs=1000 # Epochs in training stage.
lr=0.001 # Learning rate

"""### Step 4: Define SRNN and Inference Networks"""

model = model_srnn.Model(input_shape,num_tv,hidden_shape).to(device)
rnninfer=inference_network.RNNInfer(input_shape,hidden_shape).to(device)


"""### Step 5: Initialization"""

optimizer = torch.optim.Adam(list(model.parameters())+list(rnninfer.parameters()) ,lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.8)



In [2]:
model_ini,rnninfer_ini,mse_all_ini,error_all_ini,mse_all_test_ini,error_all_test_ini,loss_all_ini,pos_test_all_ini=initialization.run(model,
                                                                                                                     rnninfer,
                                                                                                                     optimizer,
                                                                                                                     scheduler,
                                                                                                                     X_train,
                                                                                                                     y_train,
                                                                                                                     X_test,
                                                                                                                     y_test,
                                                                                                                     beh_all_train,
                                                                                                                     beh_all_test,
                                                                                                                     num_tv,
                                                                                                                     coef_cross,
                                                                                                                     ini_epochs,
                                                                                                                     device,
                                                                                                                     method='kmeans',
                                                                                                                     t_load=None)

"""Now, we can test the SRNN after initialization."""

y_pred_test_ini,pos_test_ini,sampled_h_test_ini=train.eval_(model_ini,rnninfer_ini,X_test,y_test,device)

"""#### We also include another tutorial using 'random' initailization, SRNNs are also able to identify correct states.

### Step 6: Training
"""



C:\Users\yongx\anaconda3\envs\myssm\Lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


Epoch 1/3000, loss = 67428651.05757642
Epoch 101/3000, loss = 6445878.944682777
Training Still Needs :- 00d01h27m17s  (Estimated).
Epoch 201/3000, loss = 4641397.333543003
Training Still Needs :- 00d01h24m13s  (Estimated).
Epoch 301/3000, loss = 3333076.7800866365
Training Still Needs :- 00d01h21m17s  (Estimated).
Epoch 401/3000, loss = 3018327.4712511105
Training Still Needs :- 00d01h18m11s  (Estimated).
Epoch 501/3000, loss = 2807013.7557304497
Training Still Needs :- 00d01h15m10s  (Estimated).
Epoch 601/3000, loss = 2655185.0269547403
Training Still Needs :- 00d01h11m59s  (Estimated).
Epoch 701/3000, loss = 2549796.554091409
Training Still Needs :- 00d01h09m03s  (Estimated).
Epoch 801/3000, loss = 2475725.89566617
Training Still Needs :- 00d01h06m06s  (Estimated).
Epoch 901/3000, loss = 2412631.1547434316
Training Still Needs :- 00d01h03m09s  (Estimated).
Epoch 1001/3000, loss = 2356758.153220831
Training Still Needs :- 00d01h00m12s  (Estimated).
Epoch 1101/3000, loss = 2303778.4612

"#### We also include another tutorial using 'random' initailization, SRNNs are also able to identify correct states.\n\n### Step 6: Training\n"

In [6]:
model_trained,rnninfer_trained,mse_all_train,error_all_train,mse_all_test,error_all_test,loss_all,pos_test_all=train.train_(model_ini,
                                                                                                                    rnninfer_ini,
                                                                                                                    optimizer,
                                                                                                                    scheduler,
                                                                                                                    X_train,
                                                                                                                    y_train,
                                                                                                                    X_test,
                                                                                                                    y_test,
                                                                                                                    beh_all_train,
                                                                                                                    beh_all_test,
                                                                                                                    num_tv,
                                                                                                                    epochs,
                                                                                                                    device)

"""Now, we can test the SRNN after training.

### Step 7: Analysis
"""

y_pred_test,pos_test,sampled_h_test=train.eval_(model_trained,rnninfer_trained,X_test,y_test,device)




Epoch 1/1000, loss = 1511069.375
Epoch 101/1000, loss = 1491226.375
Training Still Needs :- 00d00h29m45s  (Estimated).
Epoch 201/1000, loss = 1469561.75
Training Still Needs :- 00d00h26m27s  (Estimated).
Epoch 301/1000, loss = 1452817.25
Training Still Needs :- 00d00h23m01s  (Estimated).
Epoch 401/1000, loss = 1438182.75
Training Still Needs :- 00d00h19m44s  (Estimated).
Epoch 501/1000, loss = 1419339.25
Training Still Needs :- 00d00h16m23s  (Estimated).
Epoch 601/1000, loss = 1403534.5
Training Still Needs :- 00d00h13m05s  (Estimated).
Epoch 701/1000, loss = 1379543.125
Training Still Needs :- 00d00h09m50s  (Estimated).
Epoch 801/1000, loss = 1356617.0
Training Still Needs :- 00d00h06m32s  (Estimated).
Epoch 901/1000, loss = 1338734.0
Training Still Needs :- 00d00h03m15s  (Estimated).


In [10]:
torch.save({
            'num_tv':num_tv,
            'hidden_shape':hidden_shape,
            'y_train':y_train.cpu().detach().numpy(),
            'y_train':X_train.cpu().detach().numpy(),
            'y_test':y_test.cpu().detach().numpy(),
            'X_test':X_test.cpu().detach().numpy(),
            'mse_all_ini':mse_all_ini,
            'error_all_ini':error_all_ini,
            'mse_all_test_ini':mse_all_test_ini,
            'error_all_test_ini':error_all_test_ini,
            'loss_train_ini':loss_all_ini,
            'pos_test_all_ini':pos_test_all_ini,
            'model_state_dict': model_trained.state_dict(),
            'rnninfer_state_dict': rnninfer_trained.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'mse_all':mse_all_train,
            'error_all':error_all_train,
            'mse_all_test':mse_all_test,
            'error_all_test':error_all_test,
            'loss_train':loss_all,
            'pos_test_all':pos_test_all,
            }, './result/model_area2_'+str(jobid)+'.pt')